In [1]:
%matplotlib inline

%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('..')

In [3]:
import numpy as np
import pylab as plt
from loguru import logger

from stable_baselines3 import PPO, DQN

In [4]:
from vimms.Common import POSITIVE
from vimms.ChemicalSamplers import MZMLFormulaSampler, MZMLRTandIntensitySampler

from vimms_gym.env import DDAEnv
from vimms_gym.chemicals import generate_chemicals
from vimms_gym.evaluation import evaluate, evaluate_method
from vimms_gym.features import obs_to_dfs

# 1. Parameters

In [5]:
# n_chemicals = (200, 500)
# mz_range = (100, 600)
# rt_range = (0, 300)
# intensity_range = (1E5, 1E10)

In [6]:
n_chemicals = (500, 2000)
mz_range = (100, 600)
rt_range = (200, 1000)
intensity_range = (1E5, 1E10)

In [7]:
min_mz = mz_range[0]
max_mz = mz_range[1]
min_rt = rt_range[0]
max_rt = rt_range[1]
min_log_intensity = np.log(intensity_range[0])
max_log_intensity = np.log(intensity_range[1])

In [8]:
isolation_window = 0.7
N = 10
rt_tol = 15
mz_tol = 10
min_ms1_intensity = 5000
ionisation_mode = POSITIVE
noise_density = 0.3
noise_max_val = 1e4

In [9]:
mzml_filename = 'Beer_multibeers_1_fullscan1.mzML'
mz_sampler = MZMLFormulaSampler(mzml_filename, min_mz=min_mz, max_mz=max_mz)
ri_sampler = MZMLRTandIntensitySampler(mzml_filename, min_rt=min_rt, max_rt=max_rt,
                                       min_log_intensity=min_log_intensity,
                                       max_log_intensity=max_log_intensity)

2022-03-21 14:32:58.072 | DEBUG    | mass_spec_utils.data_import.mzml:_load_file:166 - Loaded 1109 scans
2022-03-21 14:33:00.334 | DEBUG    | mass_spec_utils.data_import.mzml:_load_file:166 - Loaded 1109 scans


In [10]:
params = {
    'chemical_creator': {
        'mz_range': mz_range,
        'rt_range': rt_range,
        'intensity_range': intensity_range,
        'n_chemicals': n_chemicals,
        'mz_sampler': mz_sampler,
        'ri_sampler': ri_sampler,
    },
    'noise': {
        'noise_density': noise_density,
        'noise_max_val': noise_max_val,
        'mz_range': mz_range
    },
    'env': {
        'ionisation_mode': ionisation_mode,
        'rt_range': rt_range,
        'isolation_window': isolation_window,
        'mz_tol': mz_tol,
        'rt_tol': rt_tol,
    }
}

# 2. Evaluation

Generate some chemical sets

In [22]:
n_eval_episodes = 1

In [23]:
chemical_creator_params = params['chemical_creator']

chem_list = []
for i in range(n_eval_episodes):
    print(i)
    chems = generate_chemicals(chemical_creator_params)
    chem_list.append(chems)

0


In [57]:
methods = [
    'fullscan',    
    'TopN',
    'random',
]
max_peaks_list = [20]
rt_tol_list = [15, 120]
out_dir = 'checking_TopN'

In [42]:
chemical_creator_params = params['chemical_creator']

chem_list = []
for i in range(1):
    print(i)
    chems = generate_chemicals(chemical_creator_params)
    chem_list.append(chems)

0


In [61]:
for max_peaks in max_peaks_list:
    for rt_tol in rt_tol_list:
        
        copy_params = dict(params)
        copy_params['env']['rt_tol'] = rt_tol
        
        for method in methods:
            banner = 'method = %s max_peaks = %d rt_tol = %d' % (method, max_peaks, rt_tol)
            print(banner)
            print()

            model = None
            if method == 'DQN':
                training_dir = 'results_%d_%d' % (max_peaks, rt_tol)
                fname = '%s/model_%s' % (training_dir, method)
                model = DQN.load(fname)                
            elif method == 'PPO':
                training_dir = 'results_%d_%d' % (max_peaks, rt_tol)
                fname = '%s/model_%s' % (training_dir, method)
                model = PPO.load(fname)                
            
            results_dir = '%s/eval_%d_%d' % (out_dir, max_peaks, rt_tol)
            env = DDAEnv(max_peaks, copy_params)
            evaluate_method(env, chem_list, method, results_dir, min_ms1_intensity=min_ms1_intensity, model=model)
            print()

method = fullscan max_peaks = 20 rt_tol = 15

Episode 0 finished after 2001 timesteps with reward -199.99999999999292
(0.0, 0.0)

method = TopN max_peaks = 20 rt_tol = 15

Episode 0 finished after 2657 timesteps with reward 296.95308692762706
(0.7278688524590164, 0.6188084146302537)

method = random max_peaks = 20 rt_tol = 15

Episode 0 finished after 3787 timesteps with reward 80.08977237870754
(0.6983606557377049, 0.6329621290514751)

method = fullscan max_peaks = 20 rt_tol = 120

Episode 0 finished after 2001 timesteps with reward -199.99999999999292
(0.0, 0.0)

method = TopN max_peaks = 20 rt_tol = 120

Episode 0 finished after 2225 timesteps with reward 193.58013887040565
(0.673224043715847, 0.30779850732930103)

method = random max_peaks = 20 rt_tol = 120

Episode 0 finished after 3798 timesteps with reward 80.57316630844994
(0.7016393442622951, 0.6357854699635856)

